# Schur Residual SOHO — single locked held-out evaluation

Run this notebook exactly once with the saved Phase F `gate_results.json` or ZIP. It verifies the artifact and source training cache before opening `test.pt`. There is no hyperparameter search or manual rank/Ridge input.

In [ ]:
# === Edit paths/source only; there are no model hyperparameters in this cell ===
REPO_GIT_URL = 'https://github.com/ZaPhat206/SOHO-CL.git'
REPO_BRANCH = 'feature/crt-soho'  # push Phase G before running
CHECKPOINT_SOURCE = 'huggingface'
DRIVE_CHECKPOINT_PATH = '/content/drive/MyDrive/T-SOHO/model.safetensors'
WORK_DIR = '/content/SOHO-CL'
FEATURE_CACHE_DIR = '/content/tsoho_cifar100_cache'
OUTPUT_DIR = '/content/schur_locked_heldout_outputs'
BATCH_SIZE = 128
CHECKPOINT_SIZE = 346284714
CHECKPOINT_SHA256 = '32aa17d6e17b43500f531d5f6dc9bc93e56ed8841b8a75682e1bb295d722405b'


In [ ]:
import hashlib, json, os, shutil, subprocess, sys, zipfile, torch
from pathlib import Path
assert torch.cuda.is_available(), 'Select Runtime > Change runtime type > T4 GPU.'
%cd /content
shutil.rmtree(WORK_DIR, ignore_errors=True)
subprocess.run(['git', 'clone', '--branch', REPO_BRANCH, REPO_GIT_URL, WORK_DIR], check=True)
%cd {WORK_DIR}
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-kaggle.txt', 'kagglehub', 'huggingface_hub'], check=True)
subprocess.run(['git', 'log', '-1', '--oneline'], check=True)
subprocess.run(['nvidia-smi'], check=True)


In [ ]:
# Upload the exact Phase F gate_results.json or schur_residual_gate_results.zip.
from google.colab import files
uploaded = files.upload()
assert len(uploaded) == 1, 'Upload exactly one JSON or ZIP artifact.'
uploaded_name, uploaded_bytes = next(iter(uploaded.items()))
GATE_RESULTS_PATH = '/content/locked_gate_results.json'
if uploaded_name.lower().endswith('.zip'):
    archive_path = Path('/content') / uploaded_name
    archive_path.write_bytes(uploaded_bytes)
    with zipfile.ZipFile(archive_path) as archive:
        matches = [name for name in archive.namelist() if name.endswith('/gate_results.json') or name == 'gate_results.json']
        assert len(matches) == 1, f'Expected one gate_results.json, found: {matches}'
        Path(GATE_RESULTS_PATH).write_bytes(archive.read(matches[0]))
elif uploaded_name.lower().endswith('.json'):
    Path(GATE_RESULTS_PATH).write_bytes(uploaded_bytes)
else:
    raise ValueError('Artifact must be gate_results.json or a ZIP containing it.')
GATE_RESULTS_SHA256 = hashlib.sha256(Path(GATE_RESULTS_PATH).read_bytes()).hexdigest()
print('Locked gate artifact:', GATE_RESULTS_PATH)
print('SHA-256:', GATE_RESULTS_SHA256)
payload = json.load(open(GATE_RESULTS_PATH))
print('status:', payload.get('status'), 'authorized:', payload.get('held_out_test_authorized'))


In [ ]:
# Reuse the exact frozen feature cache. If absent, reconstruct it; the locked runner will verify train.pt SHA-256.
if not Path(FEATURE_CACHE_DIR, 'metadata.json').is_file():
    if CHECKPOINT_SOURCE == 'google_drive':
        from google.colab import drive
        drive.mount('/content/drive')
        CHECKPOINT_PATH = DRIVE_CHECKPOINT_PATH
    else:
        from huggingface_hub import hf_hub_download
        CHECKPOINT_PATH = hf_hub_download(repo_id='timm/vit_base_patch16_224.augreg2_in21k_ft_in1k', filename='model.safetensors')
    import kagglehub
    downloaded = Path(kagglehub.dataset_download('zaphat206/cifar-100'))
    candidates = [downloaded, *downloaded.rglob('cifar-100')]
    cifar_dir = next(p for p in candidates if (p/'train').is_file() and (p/'test').is_file() and (p/'meta').is_file())
    command = [sys.executable, '-u', 'tools/experiment_runner.py', '--extract-features-only', '--root', str(cifar_dir), '--backbone-checkpoint', CHECKPOINT_PATH, '--backbone-checkpoint-size', str(CHECKPOINT_SIZE), '--backbone-checkpoint-sha256', CHECKPOINT_SHA256, '--feature-cache-dir', FEATURE_CACHE_DIR, '--output-dir', f'{OUTPUT_DIR}/feature_extract', '--dataset', 'CIFAR-100', '--model-name', 'vit_base_patch16_224', '--data-augmentation', 'vit', '--seed', '1993', '--num-classes', '100', '--num-tasks', '10', '--device', 'cuda', '--batch-size', str(BATCH_SIZE), '--num-workers', '2']
    print('Running feature extraction:', ' '.join(command), flush=True)
    subprocess.run(command, check=True)
else:
    print('Using existing frozen-feature cache:', FEATURE_CACHE_DIR)


In [ ]:
# Authorization/unit tests. These do not open your held-out cache.
subprocess.run([sys.executable, '-m', 'pytest', '-q', 'tests/test_schur_locked_eval.py', 'tests/test_crt_soho_math.py', 'tests/test_crt_gate_runner.py'], check=True)


In [ ]:
# The single locked held-out run. Config is taken from the authorized artifact.
command = [sys.executable, '-u', 'tools/schur_locked_eval.py', '--gate-results', GATE_RESULTS_PATH, '--gate-results-sha256', GATE_RESULTS_SHA256, '--feature-cache-dir', FEATURE_CACHE_DIR, '--output-dir', OUTPUT_DIR, '--dataset', 'CIFAR-100', '--model-name', 'vit_base_patch16_224', '--num-classes', '100', '--num-tasks', '10', '--seed', '1993', '--device', 'cuda', '--anchor-dim', '1024', '--synaptic-degree', '300', '--coding-level', '0.3', '--statistics-dtype', 'float32', '--anchor-batch-size', '1024']
print('Running locked evaluation:', ' '.join(command), flush=True)
subprocess.run(command, check=True)


In [ ]:
# Display and download the immutable evidence bundle. Do not tune from this table.
import pandas as pd
result = json.load(open(f'{OUTPUT_DIR}/heldout_results.json'))
columns = ['method', 'final_accuracy', 'average_incremental_accuracy', 'forgetting', 'persistent_state_bytes', 'final_effective_rank', 'solver_relative_residual_max', 'retained_correction_energy', 'classifier_solve_seconds', 'classifier_recompute_seconds', 'total_inference_seconds']
table = pd.DataFrame(result['results'])
display(table[[column for column in columns if column in table]].sort_values('average_incremental_accuracy', ascending=False))
print('locked gate SHA-256:', result['lock']['gate_results_sha256'])
archive = '/content/schur_locked_heldout_results.zip'
subprocess.run(['zip', '-r', archive, OUTPUT_DIR], check=True)
files.download(archive)
